In [1]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.9 MB/s eta 0:00:00


In [2]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [3]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_217bc86399661a10f56435977bcbf87a"


In [4]:
import os, json

kaggle_creds = {
    "username": "iroonman",
    "key": "KGAT_217bc86399661a10f56435977bcbf87a"
}

os.makedirs("/root/.kaggle", exist_ok=True)

with open("/root/.kaggle/kaggle.json", "w") as f:
    json.dump(kaggle_creds, f)

os.chmod("/root/.kaggle/kaggle.json", 600)

print("kaggle.json created successfully")


kaggle.json created successfully


In [5]:
!kaggle datasets download -d andrewmvd/car-plate-detection

Dataset URL: https://www.kaggle.com/datasets/andrewmvd/car-plate-detection
License(s): CC0-1.0
 98% 200M/203M [00:00<00:00, 387MB/s]
100% 203M/203M [00:00<00:00, 419MB/s]


In [6]:
!unzip -n car-plate-detection.zip


Archive:  car-plate-detection.zip
  inflating: annotations/Cars0.xml   
  inflating: annotations/Cars1.xml   
  inflating: annotations/Cars10.xml  
  inflating: annotations/Cars100.xml  
  inflating: annotations/Cars101.xml  
  inflating: annotations/Cars102.xml  
  inflating: annotations/Cars103.xml  
  inflating: annotations/Cars104.xml  
  inflating: annotations/Cars105.xml  
  inflating: annotations/Cars106.xml  
  inflating: annotations/Cars107.xml  
  inflating: annotations/Cars108.xml  
  inflating: annotations/Cars109.xml  
  inflating: annotations/Cars11.xml  
  inflating: annotations/Cars110.xml  
  inflating: annotations/Cars111.xml  
  inflating: annotations/Cars112.xml  
  inflating: annotations/Cars113.xml  
  inflating: annotations/Cars114.xml  
  inflating: annotations/Cars115.xml  
  inflating: annotations/Cars116.xml  
  inflating: annotations/Cars117.xml  
  inflating: annotations/Cars118.xml  
  inflating: annotations/Cars119.xml  
  inflating: annotations/Cars12.xm

In [7]:
!rm car-plate-detection.zip

In [8]:
!ls annotations | head


Cars0.xml
Cars100.xml
Cars101.xml
Cars102.xml
Cars103.xml
Cars104.xml
Cars105.xml
Cars106.xml
Cars107.xml
Cars108.xml


In [9]:
import os
import xml.etree.ElementTree as ET
from shutil import copyfile
from sklearn.model_selection import train_test_split

# Paths
IMG_DIR = "images"
ANN_DIR = "annotations"
OUT_DIR = "dataset"

# Create YOLO folder structure
for split in ["train", "val"]:
    os.makedirs(f"{OUT_DIR}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUT_DIR}/labels/{split}", exist_ok=True)

classes = ["car", "licence"]

def convert_bbox(size, box):
    w, h = size
    xmin, xmax, ymin, ymax = box
    x_center = (xmin + xmax) / 2.0 / w
    y_center = (ymin + ymax) / 2.0 / h
    bw = (xmax - xmin) / w
    bh = (ymax - ymin) / h
    return x_center, y_center, bw, bh

xml_files = sorted(os.listdir(ANN_DIR))
train_files, val_files = train_test_split(xml_files, test_size=0.2, random_state=42)

def process(files, split):
    for xml_file in files:
        tree = ET.parse(os.path.join(ANN_DIR, xml_file))
        root = tree.getroot()

        img_name = root.find("filename").text
        img_path = os.path.join(IMG_DIR, img_name)

        size = root.find("size")
        w = int(size.find("width").text)
        h = int(size.find("height").text)

        label_path = os.path.join(OUT_DIR, "labels", split, img_name.replace(".png", ".txt"))

        with open(label_path, "w") as f:
            for obj in root.findall("object"):
                cls = obj.find("name").text
                cls_id = classes.index(cls)

                bbox = obj.find("bndbox")
                xmin = float(bbox.find("xmin").text)
                ymin = float(bbox.find("ymin").text)
                xmax = float(bbox.find("xmax").text)
                ymax = float(bbox.find("ymax").text)

                bb = convert_bbox((w, h), (xmin, xmax, ymin, ymax))
                f.write(f"{cls_id} {' '.join(map(str, bb))}\n")

        copyfile(img_path, os.path.join(OUT_DIR, "images", split, img_name))

process(train_files, "train")
process(val_files, "val")

print("✅ Conversion complete")


✅ Conversion complete


In [10]:
!ls dataset/images/train | head
!ls dataset/labels/train | head

Cars100.png
Cars101.png
Cars102.png
Cars103.png
Cars104.png
Cars105.png
Cars107.png
Cars108.png
Cars109.png
Cars10.png
Cars100.txt
Cars101.txt
Cars102.txt
Cars103.txt
Cars104.txt
Cars105.txt
Cars107.txt
Cars108.txt
Cars109.txt
Cars10.txt


In [11]:
%%writefile data.yaml
path: dataset
train: images/train
val: images/val

nc: 2
names: ["car", "licence"]


Writing data.yaml


In [12]:
!cat data.yaml


path: dataset
train: images/train
val: images/val

nc: 2
names: ["car", "licence"]


In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # small & fast
model.train(
    data="data.yaml",
    epochs=30,
    imgsz=640,
    batch=16
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.14 🚀 Python-3.12.12 torch-2.9.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, ke

In [ ]:
model = YOLO("runs/detect/train/weights/best.pt")
model.predict(source="car.jpeg", conf=0.4, save=True)


In [ ]:
from PIL import Image
Image.open("car.jpeg")


In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO("runs/detect/train/weights/best.pt")
results = model("car.jpeg", conf=0.4)

img = cv2.imread("car.jpeg")

plate_img = None

for r in results:
    for box in r.boxes:
        cls_id = int(box.cls[0])
        cls_name = r.names[cls_id]

        if cls_name == "licence":
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            plate_img = img[y1:y2, x1:x2]


In [ ]:
from matplotlib import pyplot as plt

plt.imshow(cv2.cvtColor(plate_img, cv2.COLOR_BGR2RGB))
plt.axis("off")


In [ ]:
!pip install easyocr

In [ ]:
import easyocr
print("EasyOCR installed successfully")


In [ ]:
import cv2
import easyocr
import numpy as np

reader = easyocr.Reader(['en'])

# Convert to grayscale
gray = cv2.cvtColor(plate_img, cv2.COLOR_BGR2GRAY)

# Resize more for small plates
gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)

# Sharpen image
kernel = np.array([[0, -1, 0],
                   [-1, 5,-1],
                   [0, -1, 0]])
sharp = cv2.filter2D(gray, -1, kernel)

# Try two threshold versions
_, thresh1 = cv2.threshold(sharp, 150, 255, cv2.THRESH_BINARY)
_, thresh2 = cv2.threshold(sharp, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

best_text = ""
best_conf = 0

for img_version in [sharp, thresh1, thresh2]:

    result = reader.readtext(
        img_version,
        allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789',
        paragraph=False,
        detail=1
    )

    full_text = ""
    confs = []

    for (_, text, prob) in result:
        full_text += text
        confs.append(prob)

    if confs:
        avg_conf = sum(confs) / len(confs)

        if avg_conf > best_conf:
            best_conf = avg_conf
            best_text = full_text

print("Final Plate:", best_text)
print("Confidence:", best_conf)


In [ ]:
import re

pattern = r'[A-Z]{2}[0-9]{2}[A-Z]{1,3}[0-9]{4}'
match = re.search(pattern, best_text)

if match:
    print(match.group())


In [ ]:
import re

def clean_plate(text):
    text = text.replace(" ", "").upper()
    match = re.findall(r'[A-Z0-9]{8,12}', text)
    return match[0] if match else text

print("Final Plate:", clean_plate(best_text))
